## SensorFusion-HAR: Hybrid Sensor Fusion for Human Activity Recognition

### Architecture Overview

SensorFusion-HAR is a lightweight model (~22.8K parameters) for wearable sensor-based human activity recognition. The pipeline consists of five stages:

1. **Echo State Network (ESN)** -- Reservoir computing layer with fixed random recurrent weights. Captures temporal dynamics without gradient-based training of recurrent connections.
2. **Depthwise Separable Convolutions (DSConv)** -- Three stacked depthwise-separable conv blocks that progressively reduce the temporal dimension while learning local patterns.
3. **Gated Residual Fusion** -- A learned gate that blends reservoir features with DSConv features via a sigmoid-gated residual connection.
4. **Patch Micro-Attention** -- Patches the feature sequence, projects to a low-dimensional space, and applies a single-layer multi-head self-attention with pre-norm and a feed-forward network.
5. **Binary Classifier** -- Batch normalization followed by a binary-weight linear layer for classification.

```
Input (B, 128, 6)
  -> ESN (B, 128, 32)
  -> transpose -> DSConv (B, 48, 32)
  -> GatedResidualFusion (B, 48, 32)
  -> PatchMicroAttention (B, 32)
  -> BinaryClassifier (B, 6)
```

## Setup and Imports

In [ ]:
import os
import sys
import copy
import time
import random
import warnings

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
plt.style.use("dark_background")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
from model import SensorFusionHAR
from model.dataset import UCIHARDataset
from model.dataset_pamap2 import PAMAP2Dataset, ACTIVITY_NAMES as PAMAP2_LABELS
from model.reservoir import EchoStateNetwork
from model.augmentation import SensorAugmentor, AugmentedDataset
from model.contrastive import SensorSimCLR, pretrain_contrastive, transfer_weights
from model.masked_pretrain import MaskedSensorModel, masked_pretrain, transfer_masked_weights
from model.multitask import MultiTaskHAR, SubjectLabeledDataset, train_multitask
from model.curriculum import CurriculumScheduler, CurriculumTrainer
from model.personalization import few_shot_personalize, evaluate_personalization
from model.adversarial import evaluate_adversarial_robustness, plot_adversarial_robustness
from model.transitions import evaluate_transition_accuracy, plot_transition_analysis
from model.drift import evaluate_drift_robustness, plot_drift_robustness
from model.energy import count_macs, estimate_energy, compare_models_energy, plot_energy_comparison
from model.visualize import plot_tsne, plot_attention_maps, plot_noise_robustness, plot_confidence_calibration
from evaluate import (
    NoReservoirModel,
    NoAttentionModel,
    NoBinaryHeadModel,
    NoDSConvModel,
    NoGateModel,
)

ACTIVITY_LABELS = ["Walking", "Upstairs", "Downstairs", "Sitting", "Standing", "Laying"]

## Dataset Loading

In [ ]:
DATA_DIR = "data/UCI HAR Dataset"

if not os.path.isdir(DATA_DIR):
    DATA_DIR = UCIHARDataset.download("data")

train_ds = UCIHARDataset(DATA_DIR, split="train")
test_ds = UCIHARDataset(DATA_DIR, split="test")

mean = train_ds.X.mean(dim=(0, 1))
std = train_ds.X.std(dim=(0, 1))
std[std < 1e-8] = 1.0

train_ds.X = (train_ds.X - mean) / std
test_ds.X = (test_ds.X - mean) / std

print(f"Train: {len(train_ds)} samples, shape {train_ds.X.shape}")
print(f"Test:  {len(test_ds)} samples, shape {test_ds.X.shape}")
print(f"Classes: {len(ACTIVITY_LABELS)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_counts = [(train_ds.y == i).sum().item() for i in range(6)]
test_counts = [(test_ds.y == i).sum().item() for i in range(6)]

x_pos = np.arange(6)
axes[0].bar(x_pos - 0.2, train_counts, 0.4, label="Train", color="#4CAF50", alpha=0.8)
axes[0].bar(x_pos + 0.2, test_counts, 0.4, label="Test", color="#2196F3", alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(ACTIVITY_LABELS, rotation=30, ha="right")
axes[0].set_ylabel("Count")
axes[0].set_title("Class Distribution")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

sample_idx = 0
sample_x = train_ds.X[sample_idx].numpy()
ch_names = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
for ch in range(6):
    axes[1].plot(sample_x[:, ch], label=ch_names[ch], alpha=0.8)
axes[1].set_xlabel("Time step")
axes[1].set_ylabel("Normalized value")
axes[1].set_title(f"Sample window (label: {ACTIVITY_LABELS[train_ds.y[sample_idx].item()]})")
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Data Augmentation Demo

In [ ]:
augmentor = SensorAugmentor(p=1.0)
sample_np = train_ds.X[0].numpy().copy()

aug_names = ["jitter", "scaling", "rotation", "permutation", "time_warp", "magnitude_warp", "channel_dropout"]
aug_fns = [
    augmentor.jitter,
    augmentor.scaling,
    augmentor.rotation,
    augmentor.permutation,
    augmentor.time_warp,
    augmentor.magnitude_warp,
    augmentor.channel_dropout,
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

axes[0].plot(sample_np[:, 0], color="#4CAF50")
axes[0].set_title("Original (ch0)")
axes[0].grid(True, alpha=0.3)

for i, (name, fn) in enumerate(zip(aug_names, aug_fns)):
    augmented = fn(sample_np.copy())
    axes[i + 1].plot(augmented[:, 0], color="#FF9800")
    axes[i + 1].set_title(name)
    axes[i + 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Model Architecture

In [ ]:
model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
print(model)
print(f"\nTotal trainable parameters: {model.count_parameters():,}")
print(f"Model size (FP32): {model.model_size_kb():.2f} KB")
print(f"Quantized size (INT8): {model.quantized_size_kb():.2f} KB")

In [ ]:
print("Per-module parameter breakdown:")
print("-" * 45)
for name, module in model.named_children():
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    total = sum(p.numel() for p in module.parameters())
    buffers = sum(b.numel() for b in module.buffers())
    print(f"  {name:<15s}  trainable={trainable:>6,}  total={total:>6,}  buffers={buffers:>6,}")

dummy = torch.randn(2, 128, 6).to(device)
out = model(dummy)
print(f"\nForward pass test: input {dummy.shape} -> output {out.shape}")

## Training with Data Augmentation

In [ ]:
augmentor_train = SensorAugmentor(p=0.5)
aug_train_ds = AugmentedDataset(train_ds, augmentor_train)

train_loader = DataLoader(aug_train_ds, batch_size=64, shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=0.001, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

os.makedirs("checkpoints", exist_ok=True)

train_losses = []
test_accs = []
test_f1s = []
best_acc = 0.0

for epoch in range(1, 101):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    scheduler.step()

    avg_loss = total_loss / max(n_batches, 1)
    train_losses.append(avg_loss)

    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
            all_preds.append(preds.cpu())
            all_labels.append(yb.cpu())

    acc = correct / total
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    f1 = f1_score(all_labels, all_preds, average="macro")
    test_accs.append(acc)
    test_f1s.append(f1)

    if acc > best_acc:
        best_acc = acc
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "accuracy": acc,
                "f1_macro": f1,
                "normalization_stats": {"means": mean.tolist(), "stds": std.tolist()},
            },
            "checkpoints/best_model.pt",
        )

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/100  loss={avg_loss:.4f}  acc={acc:.4f}  f1={f1:.4f}  best_acc={best_acc:.4f}")

print(f"\nTraining complete. Best accuracy: {best_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, color="#F44336", linewidth=1.5)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(test_accs, color="#4CAF50", linewidth=1.5, label="Accuracy")
axes[1].plot(test_f1s, color="#2196F3", linewidth=1.5, label="F1 Macro")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("Test Metrics")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluation

In [ ]:
checkpoint = torch.load("checkpoints/best_model.pt", map_location=device, weights_only=False)
model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Checkpoint accuracy: {checkpoint['accuracy']:.4f}")
print(f"Checkpoint F1 macro: {checkpoint['f1_macro']:.4f}")

In [ ]:
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        preds = model(xb).argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(yb)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=ACTIVITY_LABELS)
disp.plot(ax=axes[0], cmap="inferno", colorbar=True)
axes[0].set_title("Confusion Matrix")
axes[0].set_xticklabels(ACTIVITY_LABELS, rotation=30, ha="right")

f1_per_class = f1_score(all_labels, all_preds, average=None)
bars = axes[1].bar(ACTIVITY_LABELS, f1_per_class, color="#4CAF50", alpha=0.8, edgecolor="white")
for bar, val in zip(bars, f1_per_class):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{val:.3f}",
                 ha="center", va="bottom", fontsize=9)
axes[1].set_ylabel("F1 Score")
axes[1].set_title("Per-Class F1")
axes[1].set_ylim(0, 1.1)
axes[1].set_xticklabels(ACTIVITY_LABELS, rotation=30, ha="right")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print(classification_report(all_labels, all_preds, target_names=ACTIVITY_LABELS))

## Ablation Study

In [ ]:
ablation_variants = [
    ("Full Model", SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6)),
    ("No Reservoir", NoReservoirModel(num_classes=6)),
    ("No Attention", NoAttentionModel(num_classes=6)),
    ("No BinaryHead", NoBinaryHeadModel(num_classes=6)),
    ("No DSConv", NoDSConvModel(num_classes=6)),
    ("No Gate", NoGateModel(num_classes=6)),
]

ablation_train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)
ablation_test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

ablation_results = []

for name, variant_model in ablation_variants:
    variant_model = variant_model.to(device)
    params = variant_model.count_parameters()
    print(f"\n--- {name} ({params:,} params) ---")

    abl_optimizer = torch.optim.AdamW(
        [p for p in variant_model.parameters() if p.requires_grad], lr=0.001, weight_decay=1e-4
    )
    abl_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(abl_optimizer, T_max=50)
    abl_criterion = nn.CrossEntropyLoss()

    best_variant_acc = 0.0
    best_variant_state = None

    for epoch in range(1, 51):
        variant_model.train()
        for xb, yb in ablation_train_loader:
            xb, yb = xb.to(device), yb.to(device)
            abl_optimizer.zero_grad()
            loss = abl_criterion(variant_model(xb), yb)
            loss.backward()
            abl_optimizer.step()
        abl_scheduler.step()

        variant_model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in ablation_test_loader:
                xb, yb = xb.to(device), yb.to(device)
                correct += (variant_model(xb).argmax(1) == yb).sum().item()
                total += yb.size(0)
        acc = correct / total
        if acc > best_variant_acc:
            best_variant_acc = acc
            best_variant_state = copy.deepcopy(variant_model.state_dict())

        if epoch % 10 == 0:
            print(f"  Epoch {epoch}/50  acc={acc:.4f}")

    if best_variant_state is not None:
        variant_model.load_state_dict(best_variant_state)

    variant_model.eval()
    v_preds, v_labels = [], []
    with torch.no_grad():
        for xb, yb in ablation_test_loader:
            xb = xb.to(device)
            v_preds.append(variant_model(xb).argmax(1).cpu())
            v_labels.append(yb)
    v_preds = torch.cat(v_preds).numpy()
    v_labels = torch.cat(v_labels).numpy()
    f1 = f1_score(v_labels, v_preds, average="macro")

    ablation_results.append((name, params, best_variant_acc, f1))
    print(f"  Best: acc={best_variant_acc:.4f}  f1={f1:.4f}")

In [ ]:
print(f"{'Variant':<18s} {'Params':>8s} {'Accuracy':>10s} {'F1 Macro':>10s}")
print("-" * 48)
for name, params, acc, f1 in ablation_results:
    print(f"{name:<18s} {params:>8,} {acc:>9.4f} {f1:>10.4f}")

fig, ax = plt.subplots(figsize=(12, 5))
names = [r[0] for r in ablation_results]
accs = [r[2] for r in ablation_results]
f1s = [r[3] for r in ablation_results]
x = np.arange(len(names))
ax.bar(x - 0.2, accs, 0.4, label="Accuracy", color="#4CAF50", alpha=0.8)
ax.bar(x + 0.2, f1s, 0.4, label="F1 Macro", color="#2196F3", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title("Ablation Study Results")
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## t-SNE Feature Visualization

In [ ]:
checkpoint = torch.load("checkpoints/best_model.pt", map_location=device, weights_only=False)
model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

fig = plot_tsne(model, test_ds, device, stage="all", n_samples=1000, activity_labels=ACTIVITY_LABELS)
plt.show()

## Attention Map Visualization

In [ ]:
fig = plot_attention_maps(model, test_ds, device, n_samples=4, activity_labels=ACTIVITY_LABELS)
plt.show()

## Noise Robustness Analysis

In [ ]:
noise_results, noise_fig = plot_noise_robustness(model, test_ds, device)
plt.show()

for snr, acc, f1 in zip(noise_results["snr_levels"], noise_results["accuracy"], noise_results["f1"]):
    print(f"SNR={snr:3d} dB  Accuracy={acc:.4f}  F1={f1:.4f}")

## Confidence Calibration

In [ ]:
ece, cal_fig = plot_confidence_calibration(model, test_ds, device, n_bins=10)
plt.show()
print(f"Expected Calibration Error (ECE): {ece:.4f}")

## Contrastive Pre-Training (SimCLR)

In [ ]:
simclr_model = SensorSimCLR(input_channels=6, reservoir_size=32)
simclr_augmentor = SensorAugmentor(p=0.5)

simclr_model = pretrain_contrastive(
    simclr_model, train_ds, simclr_augmentor, device,
    epochs=50, batch_size=128, lr=0.0003, temperature=0.1
)
print("Contrastive pre-training complete.")

In [ ]:
simclr_finetune = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
simclr_finetune = transfer_weights(simclr_model, simclr_finetune)

ft_optimizer = torch.optim.AdamW(
    [p for p in simclr_finetune.parameters() if p.requires_grad], lr=0.001, weight_decay=1e-4
)
ft_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=50)
ft_criterion = nn.CrossEntropyLoss()

simclr_ft_train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)

best_simclr_acc = 0.0
for epoch in range(1, 51):
    simclr_finetune.train()
    for xb, yb in simclr_ft_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        ft_optimizer.zero_grad()
        loss = ft_criterion(simclr_finetune(xb), yb)
        loss.backward()
        ft_optimizer.step()
    ft_scheduler.step()

    simclr_finetune.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (simclr_finetune(xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    acc = correct / total
    best_simclr_acc = max(best_simclr_acc, acc)
    if epoch % 10 == 0:
        print(f"SimCLR Fine-tune Epoch {epoch}/50  acc={acc:.4f}")

print(f"SimCLR best fine-tune accuracy: {best_simclr_acc:.4f}")

## Masked Sensor Modeling (MSM)

In [ ]:
msm_model = MaskedSensorModel(input_channels=6, reservoir_size=32, mask_ratio=0.15)

msm_model = masked_pretrain(
    msm_model, train_ds, device,
    epochs=50, batch_size=128, lr=0.0003, mask_ratio=0.15
)
print("Masked sensor pre-training complete.")

In [ ]:
msm_finetune = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
msm_finetune = transfer_masked_weights(msm_model, msm_finetune)

msm_optimizer = torch.optim.AdamW(
    [p for p in msm_finetune.parameters() if p.requires_grad], lr=0.001, weight_decay=1e-4
)
msm_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(msm_optimizer, T_max=50)
msm_criterion = nn.CrossEntropyLoss()

msm_ft_train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)

best_msm_acc = 0.0
for epoch in range(1, 51):
    msm_finetune.train()
    for xb, yb in msm_ft_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        msm_optimizer.zero_grad()
        loss = msm_criterion(msm_finetune(xb), yb)
        loss.backward()
        msm_optimizer.step()
    msm_scheduler.step()

    msm_finetune.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (msm_finetune(xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    acc = correct / total
    best_msm_acc = max(best_msm_acc, acc)
    if epoch % 10 == 0:
        print(f"MSM Fine-tune Epoch {epoch}/50  acc={acc:.4f}")

print(f"MSM best fine-tune accuracy: {best_msm_acc:.4f}")

In [ ]:
print("Pre-training comparison:")
print(f"  Supervised only:    {best_acc:.4f}")
print(f"  SimCLR + fine-tune: {best_simclr_acc:.4f}")
print(f"  MSM + fine-tune:    {best_msm_acc:.4f}")

## Multi-Task Adversarial Training

In [ ]:
train_subjects = np.loadtxt(
    os.path.join(DATA_DIR, "train", "subject_train.txt"), dtype=int
)
test_subjects = np.loadtxt(
    os.path.join(DATA_DIR, "test", "subject_test.txt"), dtype=int
)

all_subj = np.concatenate([train_subjects, test_subjects])
unique_subj = sorted(set(all_subj))
subj_map = {s: i for i, s in enumerate(unique_subj)}
num_subjects = len(unique_subj)

mapped_train_subj = torch.tensor([subj_map[s] for s in train_subjects], dtype=torch.long)
mapped_test_subj = torch.tensor([subj_map[s] for s in test_subjects], dtype=torch.long)

print(f"Total unique subjects: {num_subjects}")

mt_train_ds = SubjectLabeledDataset(train_ds.X, train_ds.y, mapped_train_subj)
mt_test_ds = SubjectLabeledDataset(test_ds.X, test_ds.y, mapped_test_subj)

mt_model = MultiTaskHAR(
    input_channels=6, reservoir_size=32, num_classes=6, num_subjects=num_subjects
)

mt_model = train_multitask(
    mt_model, mt_train_ds, mt_test_ds, device,
    epochs=100, batch_size=64, lr=0.001, lambda_schedule="linear"
)

mt_backbone = mt_model.extract_backbone()
mt_backbone = mt_backbone.to(device)
mt_backbone.eval()

correct, total = 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        correct += (mt_backbone(xb).argmax(1) == yb).sum().item()
        total += yb.size(0)
mt_acc = correct / total
print(f"Multi-task backbone accuracy: {mt_acc:.4f}")

## Curriculum Learning

In [ ]:
curriculum_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
cur_scheduler = CurriculumScheduler(dataset_name="ucihar", total_epochs=100, num_phases=3)
trainer = CurriculumTrainer(
    curriculum_model, train_ds, test_ds, device, cur_scheduler, batch_size=64, lr=0.001
)

curriculum_model, cur_history = trainer.train(epochs=100)

print(f"\nCurriculum final accuracy: {cur_history['test_acc'][-1]:.4f}")
print(f"Best curriculum accuracy:  {max(cur_history['test_acc']):.4f}")
print(f"Standard training best:    {best_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cur_history["train_loss"], color="#F44336", linewidth=1.5)
for phase_boundary in [cur_scheduler.phase_epochs, 2 * cur_scheduler.phase_epochs]:
    axes[0].axvline(x=phase_boundary, color="#FF9800", linestyle="--", alpha=0.7)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Curriculum Training Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(cur_history["test_acc"], color="#4CAF50", linewidth=1.5)
for phase_boundary in [cur_scheduler.phase_epochs, 2 * cur_scheduler.phase_epochs]:
    axes[1].axvline(x=phase_boundary, color="#FF9800", linestyle="--", alpha=0.7)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Curriculum Test Accuracy")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## LOSO Cross-Validation

In [ ]:
subjects = UCIHARDataset.get_subjects(DATA_DIR)
loso_accs = []
loso_f1s = []

for subj in subjects:
    loso_train, loso_test = UCIHARDataset.loso_split(DATA_DIR, subj)

    loso_mean = loso_train.X.mean(dim=(0, 1))
    loso_std = loso_train.X.std(dim=(0, 1))
    loso_std[loso_std < 1e-8] = 1.0
    loso_train.X = (loso_train.X - loso_mean) / loso_std
    loso_test.X = (loso_test.X - loso_mean) / loso_std

    loso_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
    loso_opt = torch.optim.AdamW(
        [p for p in loso_model.parameters() if p.requires_grad], lr=0.001, weight_decay=1e-4
    )
    loso_sched = torch.optim.lr_scheduler.CosineAnnealingLR(loso_opt, T_max=50)
    loso_crit = nn.CrossEntropyLoss()
    loso_train_loader = DataLoader(loso_train, batch_size=64, shuffle=True, drop_last=len(loso_train) >= 64)
    loso_test_loader = DataLoader(loso_test, batch_size=64, shuffle=False)

    best_subj_acc = 0.0
    best_subj_state = None
    for epoch in range(1, 51):
        loso_model.train()
        for xb, yb in loso_train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loso_opt.zero_grad()
            loss = loso_crit(loso_model(xb), yb)
            loss.backward()
            loso_opt.step()
        loso_sched.step()

        loso_model.eval()
        c, t = 0, 0
        with torch.no_grad():
            for xb, yb in loso_test_loader:
                xb, yb = xb.to(device), yb.to(device)
                c += (loso_model(xb).argmax(1) == yb).sum().item()
                t += yb.size(0)
        subj_acc = c / t
        if subj_acc > best_subj_acc:
            best_subj_acc = subj_acc
            best_subj_state = copy.deepcopy(loso_model.state_dict())

    if best_subj_state is not None:
        loso_model.load_state_dict(best_subj_state)
    loso_model.eval()
    loso_preds, loso_labels = [], []
    with torch.no_grad():
        for xb, yb in loso_test_loader:
            xb = xb.to(device)
            loso_preds.append(loso_model(xb).argmax(1).cpu())
            loso_labels.append(yb)
    loso_preds = torch.cat(loso_preds).numpy()
    loso_labels = torch.cat(loso_labels).numpy()
    subj_f1 = f1_score(loso_labels, loso_preds, average="macro")

    loso_accs.append(best_subj_acc)
    loso_f1s.append(subj_f1)
    print(f"Subject {subj:2d}:  acc={best_subj_acc:.4f}  f1={subj_f1:.4f}")

print(f"\nLOSO Accuracy: {np.mean(loso_accs):.4f} +/- {np.std(loso_accs):.4f}")
print(f"LOSO F1 Macro: {np.mean(loso_f1s):.4f} +/- {np.std(loso_f1s):.4f}")

## Few-Shot Personalization

In [ ]:
checkpoint = torch.load("checkpoints/best_model.pt", map_location=device, weights_only=False)
base_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
base_model.load_state_dict(checkpoint["model_state_dict"])
base_model.eval()

k_shots_list = [5, 10, 20, 50]
personalization_results = {k: [] for k in k_shots_list}

test_subjects_unique = sorted(set(test_subjects.tolist()))

for subj in test_subjects_unique[:5]:
    subj_mask = test_subjects == subj
    subj_X = test_ds.X[subj_mask]
    subj_y = test_ds.y[subj_mask]

    n = len(subj_y)
    split = n // 2
    support_X, support_y = subj_X[:split], subj_y[:split]
    query_X, query_y = subj_X[split:], subj_y[split:]

    for k in k_shots_list:
        personalized = few_shot_personalize(
            base_model, support_X, support_y, device, k_shots=k
        )
        personalized.eval()
        with torch.no_grad():
            preds = personalized(query_X.to(device)).argmax(1).cpu()
        acc = (preds == query_y).float().mean().item()
        personalization_results[k].append(acc)
    print(f"Subject {subj} done")

fig, ax = plt.subplots(figsize=(10, 6))
means = [np.mean(personalization_results[k]) for k in k_shots_list]
stds = [np.std(personalization_results[k]) for k in k_shots_list]
ax.errorbar(k_shots_list, means, yerr=stds, fmt="o-", color="#4CAF50", linewidth=2,
            markersize=8, capsize=5, label="Personalized")
ax.set_xlabel("k (shots per class)")
ax.set_ylabel("Accuracy")
ax.set_title("Few-Shot Personalization")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for k in k_shots_list:
    print(f"k={k:3d}:  acc={np.mean(personalization_results[k]):.4f} +/- {np.std(personalization_results[k]):.4f}")

## Adversarial Robustness

In [ ]:
checkpoint = torch.load("checkpoints/best_model.pt", map_location=device, weights_only=False)
model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

adv_results = evaluate_adversarial_robustness(
    model, test_ds, device,
    epsilons=[0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5],
    attack="both"
)

fig = plot_adversarial_robustness(adv_results)
plt.show()

for eps, fgsm, pgd in zip(adv_results["epsilons"], adv_results["fgsm_accuracy"], adv_results["pgd_accuracy"]):
    print(f"eps={eps:.2f}  FGSM={fgsm:.4f}  PGD={pgd:.4f}")

## Activity Transition Detection

In [ ]:
transition_results = evaluate_transition_accuracy(model, test_ds, device)

print(f"Stable accuracy:     {transition_results['stable_accuracy']:.4f} (n={transition_results['stable_count']})")
print(f"Transition accuracy: {transition_results['transition_accuracy']:.4f} (n={transition_results['transition_count']})")
print(f"Overall accuracy:    {transition_results['overall_accuracy']:.4f}")

fig = plot_transition_analysis(transition_results, activity_labels=ACTIVITY_LABELS)
plt.show()

## Sensor Drift Simulation

In [ ]:
drift_results = evaluate_drift_robustness(model, test_ds, device)

fig = plot_drift_robustness(drift_results)
plt.show()

print(f"Clean accuracy: {drift_results['clean_accuracy']:.4f}")
for dtype in drift_results["drift_types"]:
    accs = drift_results[f"{dtype}_accuracy"]
    print(f"{dtype}: min_acc={min(accs):.4f}  final_acc={accs[-1]:.4f}")

## Energy Estimation

In [ ]:
energy_models = {
    "SensorFusionHAR": SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6),
    "No Reservoir": NoReservoirModel(num_classes=6),
    "No Attention": NoAttentionModel(num_classes=6),
    "No Gate": NoGateModel(num_classes=6),
}

energy_comparison = compare_models_energy(energy_models, input_shape=(1, 128, 6))

fig = plot_energy_comparison(energy_comparison)
plt.show()

print(f"{'Model':<20s} {'MACs':>12s} {'Params':>10s} {'FP32 (mJ)':>12s} {'INT8 (mJ)':>12s}")
print("-" * 68)
for name, info in energy_comparison.items():
    print(f"{name:<20s} {info['total_macs']:>12,} {info['total_params']:>10,} {info['energy_fp32_mj']:>12.6f} {info['energy_int8_mj']:>12.6f}")

## Cross-Dataset Transfer (UCI-HAR to PAMAP2)

In [ ]:
PAMAP2_DIR = "data/PAMAP2_Dataset"

if not os.path.isdir(PAMAP2_DIR):
    PAMAP2_DIR = PAMAP2Dataset.download("data")

pamap2_train = PAMAP2Dataset(PAMAP2_DIR, split="train")
pamap2_test = PAMAP2Dataset(PAMAP2_DIR, split="test")

p_mean = pamap2_train.X.mean(dim=(0, 1))
p_std = pamap2_train.X.std(dim=(0, 1))
p_std[p_std < 1e-8] = 1.0
pamap2_train.X = (pamap2_train.X - p_mean) / p_std
pamap2_test.X = (pamap2_test.X - p_mean) / p_std

pamap2_num_classes = len(torch.unique(pamap2_train.y))
print(f"PAMAP2 train: {len(pamap2_train)}, test: {len(pamap2_test)}, classes: {pamap2_num_classes}")

In [ ]:
checkpoint = torch.load("checkpoints/best_model.pt", map_location=device, weights_only=False)
source_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
source_model.load_state_dict(checkpoint["model_state_dict"])

transfer_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=pamap2_num_classes).to(device)
transfer_model.reservoir.load_state_dict(source_model.reservoir.state_dict())
transfer_model.dsconv.load_state_dict(source_model.dsconv.state_dict())
transfer_model.attention.load_state_dict(source_model.attention.state_dict())

scratch_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=pamap2_num_classes).to(device)

pamap2_train_loader = DataLoader(pamap2_train, batch_size=64, shuffle=True, drop_last=True)
pamap2_test_loader = DataLoader(pamap2_test, batch_size=64, shuffle=False)

results_transfer = {"transfer": [], "scratch": []}

for tag, m in [("transfer", transfer_model), ("scratch", scratch_model)]:
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=0.001, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
    crit = nn.CrossEntropyLoss()

    for epoch in range(1, 51):
        m.train()
        for xb, yb in pamap2_train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(m(xb), yb).backward()
            opt.step()
        sch.step()

        m.eval()
        c, t = 0, 0
        with torch.no_grad():
            for xb, yb in pamap2_test_loader:
                xb, yb = xb.to(device), yb.to(device)
                c += (m(xb).argmax(1) == yb).sum().item()
                t += yb.size(0)
        results_transfer[tag].append(c / t)

    print(f"{tag}: best_acc={max(results_transfer[tag]):.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(results_transfer["transfer"], color="#4CAF50", linewidth=1.5, label="Transfer (UCI-HAR pretrained)")
ax.plot(results_transfer["scratch"], color="#F44336", linewidth=1.5, label="From scratch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Cross-Dataset Transfer: UCI-HAR to PAMAP2")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Spectral Reservoir Initialization

In [ ]:
spectral_esn = EchoStateNetwork.spectral_init(
    train_ds, input_channels=6, reservoir_size=32, spectral_radius=0.9
)

spectral_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
spectral_model.reservoir = spectral_esn.to(device)

random_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)

spectral_results = {"spectral": [], "random": []}

for tag, m in [("spectral", spectral_model), ("random", random_model)]:
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=0.001, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
    crit = nn.CrossEntropyLoss()

    sp_train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)

    for epoch in range(1, 51):
        m.train()
        for xb, yb in sp_train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(m(xb), yb).backward()
            opt.step()
        sch.step()

        m.eval()
        c, t = 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                c += (m(xb).argmax(1) == yb).sum().item()
                t += yb.size(0)
        spectral_results[tag].append(c / t)

    print(f"{tag} init best acc: {max(spectral_results[tag]):.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(spectral_results["spectral"], color="#4CAF50", linewidth=1.5, label="Spectral Init")
ax.plot(spectral_results["random"], color="#F44336", linewidth=1.5, label="Random Init")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Reservoir Initialization Comparison")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Baseline Comparison and Pareto Plot

In [ ]:
baselines = {
    "SensorFusionHAR": {"params": model.count_parameters(), "acc": best_acc},
}

for name, params, acc, f1 in ablation_results:
    baselines[name] = {"params": params, "acc": acc}

baselines["SimCLR+FT"] = {"params": model.count_parameters(), "acc": best_simclr_acc}
baselines["MSM+FT"] = {"params": model.count_parameters(), "acc": best_msm_acc}
baselines["MultiTask"] = {"params": sum(p.numel() for p in mt_model.parameters() if p.requires_grad), "acc": mt_acc}

fig, ax = plt.subplots(figsize=(12, 7))
colors_pareto = ["#4CAF50", "#FF9800", "#2196F3", "#9C27B0", "#F44336", "#00BCD4",
                 "#FFEB3B", "#E91E63", "#8BC34A"]

for i, (name, info) in enumerate(baselines.items()):
    c = colors_pareto[i % len(colors_pareto)]
    ax.scatter(info["params"], info["acc"], c=c, s=120, zorder=5, edgecolors="white")
    ax.annotate(name, (info["params"], info["acc"]), textcoords="offset points",
                xytext=(8, 5), fontsize=8)

ax.set_xlabel("Parameters")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs Parameters (Pareto Front)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ONNX Export

In [ ]:
checkpoint = torch.load("checkpoints/best_model.pt", map_location="cpu", weights_only=False)
export_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6)
export_model.load_state_dict(checkpoint["model_state_dict"])
export_model.eval()

dummy_input = torch.randn(1, 128, 6)
onnx_path = "checkpoints/sensorfusion_har.onnx"

torch.onnx.export(
    export_model,
    dummy_input,
    onnx_path,
    input_names=["sensor_input"],
    output_names=["activity_logits"],
    dynamic_axes={"sensor_input": {0: "batch"}, "activity_logits": {0: "batch"}},
    opset_version=14,
)

onnx_size_kb = os.path.getsize(onnx_path) / 1024
print(f"ONNX model exported to: {onnx_path}")
print(f"ONNX file size: {onnx_size_kb:.2f} KB")

## Inference Benchmark

In [ ]:
checkpoint = torch.load("checkpoints/best_model.pt", map_location=device, weights_only=False)
bench_model = SensorFusionHAR(input_channels=6, reservoir_size=32, num_classes=6).to(device)
bench_model.load_state_dict(checkpoint["model_state_dict"])
bench_model.eval()

sample_input = torch.randn(1, 128, 6).to(device)

for _ in range(50):
    with torch.no_grad():
        bench_model(sample_input)

if device.type == "cuda":
    torch.cuda.synchronize()

latencies = []
for _ in range(1000):
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        bench_model(sample_input)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000)

latencies = np.array(latencies)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(latencies, bins=50, color="#4CAF50", alpha=0.8, edgecolor="white")
axes[0].axvline(x=np.mean(latencies), color="#F44336", linestyle="--", linewidth=2, label=f"Mean: {np.mean(latencies):.3f} ms")
axes[0].axvline(x=np.percentile(latencies, 95), color="#FF9800", linestyle="--", linewidth=2, label=f"P95: {np.percentile(latencies, 95):.3f} ms")
axes[0].set_xlabel("Latency (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Latency Distribution (1000 runs)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(latencies, color="#2196F3", alpha=0.5, linewidth=0.5)
axes[1].axhline(y=np.mean(latencies), color="#F44336", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Run")
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Per-Run Latency")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean:   {np.mean(latencies):.3f} ms")
print(f"Std:    {np.std(latencies):.3f} ms")
print(f"Median: {np.median(latencies):.3f} ms")
print(f"P95:    {np.percentile(latencies, 95):.3f} ms")
print(f"P99:    {np.percentile(latencies, 99):.3f} ms")
print(f"Min:    {np.min(latencies):.3f} ms")
print(f"Max:    {np.max(latencies):.3f} ms")

## Final Summary Table

In [ ]:
macs_info = count_macs(bench_model, input_shape=(1, 128, 6))
energy_fp32 = estimate_energy(macs_info, precision="fp32")
energy_int8 = estimate_energy(macs_info, precision="int8")

print("=" * 60)
print("SENSORFUSION-HAR SUMMARY")
print("=" * 60)
print(f"")
print(f"Model")
print(f"  Parameters:          {bench_model.count_parameters():,}")
print(f"  Model size (FP32):   {bench_model.model_size_kb():.2f} KB")
print(f"  Quantized (INT8):    {bench_model.quantized_size_kb():.2f} KB")
print(f"  ONNX size:           {onnx_size_kb:.2f} KB")
print(f"  Total MACs:          {macs_info['total']:,}")
print(f"  Energy FP32:         {energy_fp32['total_millijoules']:.6f} mJ")
print(f"  Energy INT8:         {energy_int8['total_millijoules']:.6f} mJ")
print(f"")
print(f"Performance (UCI-HAR)")
print(f"  Supervised:          {best_acc:.4f}")
print(f"  SimCLR + fine-tune:  {best_simclr_acc:.4f}")
print(f"  MSM + fine-tune:     {best_msm_acc:.4f}")
print(f"  Multi-task:          {mt_acc:.4f}")
print(f"  Curriculum:          {max(cur_history['test_acc']):.4f}")
print(f"  LOSO mean:           {np.mean(loso_accs):.4f} +/- {np.std(loso_accs):.4f}")
print(f"")
print(f"Transfer (UCI-HAR -> PAMAP2)")
print(f"  With transfer:       {max(results_transfer['transfer']):.4f}")
print(f"  From scratch:        {max(results_transfer['scratch']):.4f}")
print(f"")
print(f"Reservoir Init")
print(f"  Spectral init best:  {max(spectral_results['spectral']):.4f}")
print(f"  Random init best:    {max(spectral_results['random']):.4f}")
print(f"")
print(f"Latency ({device})")
print(f"  Mean:                {np.mean(latencies):.3f} ms")
print(f"  P95:                 {np.percentile(latencies, 95):.3f} ms")
print(f"  P99:                 {np.percentile(latencies, 99):.3f} ms")
print("=" * 60)